In [4]:
import torch
import torch.nn as nn

## Basic

### Define CUDA Graph

To capture a model training step with CUDA Graph in PyTorch, we need:
- the model
- static input and output tensors (in inference) or target (in training)

In [5]:
# Define model
D_in = 1024
H = 4096
D_out = 1024

model = torch.nn.Sequential(
    torch.nn.Linear(D_in, H), # (d_in, hidden_dim)
    torch.nn.ReLU(),
    torch.nn.Linear(H, D_out)
).cuda()
loss_fn = torch.nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

In [6]:
# Define static input and output
B = 16 # batch_size
static_input = torch.randn(B, D_in, device='cuda')
static_target = torch.randn(B, D_out, device='cuda')

### Capture Phase

In [8]:
# warmup in side stream
s = torch.cuda.Stream()
s.wait_stream(torch.cuda.current_stream())

with torch.cuda.stream(s):
    for i in range(3):
        optimizer.zero_grad(set_to_none=True)
        y_pred = model(static_input)
        loss = loss_fn(y_pred, static_target)
        loss.backward()
        optimizer.step()
        
torch.cuda.current_stream().wait_stream(s)

# real capture
g = torch.cuda.CUDAGraph()
optimizer.zero_grad(set_to_none=True)

with torch.cuda.graph(g):
    static_y_pred = model(static_input)
    static_loss = loss_fn(static_y_pred, static_target)
    static_loss.backward()
    optimizer.step()
    
# The captured CUDA Graph contains the CUDA operations launched by:
#   - forward pass: model(static_input)
#   - loss computation: loss_fn(static_y_pred, static_target)
#   - backward pass: static_loss.backward()
#   - optimizer update: optimizer.step()
#
# The graph reuses the same memory addresses, so static_input and
# static_target should be updated in-place before each replay.

### Reuse phase

In [9]:
real_inputs = [
    torch.rand_like(static_input) for _ in range(10)
]

real_targets = [
    torch.rand_like(static_target) for _ in range(10)
]

for data, target in zip(real_inputs, real_targets):
    # Copy data to the recorded memory addresses of the graph
    static_input.copy_(data)
    static_target.copy_(target)
    
    g.replay()
    # Params have been updated. static_y_pred, static_loss, and .grad
    # attributes hold values from computing on this iteration's data.

## Inference Example - With Different Input Batch Size

In [ ]:
# Define model
D_in = 1024
H = 4096
D_out = 1024

model = torch.nn.Sequential(
    torch.nn.Linear(D_in, H), # (d_in, hidden_dim)
    torch.nn.ReLU(),
    torch.nn.Linear(H, D_out)
).cuda()

In [12]:
# Define input and output for warmup
warmup_B = 16 # batch_size
warmup_input = torch.randn(warmup_B, D_in, device='cuda')

s = torch.cuda.Stream()
s.wait_stream(torch.cuda.current_stream())

with torch.cuda.stream(s):
    for _ in range(3):
        _ = model(warmup_input)
        
torch.cuda.current_stream().wait_stream(s)

In [16]:
candidate_bs = [1, 2, 4, 8, 16]
graph_pool = {}

for bs in candidate_bs:
    g = torch.cuda.CUDAGraph()
    static_input = torch.randn(bs, D_in, device='cuda')
    static_output = torch.empty(bs, D_out, device='cuda')
    
    with torch.cuda.graph(g):
        static_output.copy_(model(static_input))
        
        graph_pool[bs] = (g, static_input, static_output) # for reuse

In [ ]:
import random

num_iterations = 10
for i in range(num_iterations):
    bs = random.choice(candidate_bs)
    graph, static_input, static_output = graph_pool[bs]
    
    input_tensor = torch.randn(bs, D_in, device='cuda')
    static_input.copy_(input_tensor)
    graph.replay()
    # static_output generated
    
    output_tensor = static_output

## Dynamic Parameters in Model

In some cases, we need dynamic parameters in the model (such as reusing KV Cache in modern vLLM systems), however CUDA Graph records static tensor memory. How to implement this?

In [ ]:
class MyModel(nn.Module):
    def __init__(self, D_in, H, D_out):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(D_in, H),
            nn.ReLU(),
            nn.Linear(H, D_out),
        )

    def forward(self, x, scale):
        return self.net(x) * scale # <--- Dynamic value of scale

In [ ]:
model = MyModel(D_in, H, D_out).cuda()
static_input = torch.randn(bs, D_in, device="cuda")
static_scale = torch.empty((), device="cuda")  # scalar tensor <--- record the pos
static_output = torch.empty(bs, D_out, device="cuda")

# We ignore the warmup phase

g = torch.cuda.CUDAGraph()

with torch.cuda.graph(g):
    static_output.copy_(model(static_input, static_scale))

graph_pool[bs] = (g, static_input, static_scale, static_output)

In [24]:
graph, static_input, static_scale, static_output = graph_pool[bs]

new_input = torch.randn(bs, D_in, device='cuda')
static_input.copy_(new_input)
static_scale.copy_(torch.tensor(0.7, device="cuda"))

graph.replay()

output = static_output